In [1]:
from dotenv import load_dotenv
load_dotenv()

%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3
# %matplotlib inline
# %matplotlib qt5
# import mne
# mne.viz.set_browser_backend("qt")  # or "matplotlib"
# mne.set_config("MNE_BROWSER_BACKEND", "qt")  # or "matplotlib"
%gui qt

from copy import deepcopy
import time
import xarray as xr # Assuming you're using this
import numpy as np   # For the example
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import xarray as xr
# import zarr
import panel as pn
from typing import Dict, List, Tuple, Optional, Callable, Union, Any
# import holoviews as hv
# hv.extension('bokeh', logo=False)

# import hvplot.xarray
# import hvplot.pandas
# This line is crucial for displaying plots in a notebook
# hvplot.extension('bokeh') # You can also use 'matplotlib' or 'plotly'

# hv.extension('bokeh')
# hv.extension('matplotlib') # or 'matplotlib'
# hv.extension('plotly') # or 'matplotlib'
# from holoviews import opts
import panel as pn
pn.extension()

import IPython

# Jupyter-lab enable printing for any line on its own (instead of just the last one in the cell)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pyqtgraph as pg
from phopymnehelper.historical_data import HistoricalData
from pypho_timeline.widgets import SimpleTimelineWidget
from pypho_timeline.__main__ import VideoTrackDatasource

from pypho_timeline.xdf_session_discovery import discover_xdf_files_for_timeline
from pypho_timeline.timeline_builder import TimelineBuilder

def get_now_time_str(time_separator='-') -> str:
    return str(time.strftime(f"%Y-%m-%d_%H{time_separator}%m", time.localtime(time.time())))

# Create Qt application
app = pg.mkQApp("pyPhoTimelineTestingNotebookExample")
builder: TimelineBuilder = TimelineBuilder()

Automatic pdb calling has been turned OFF


c:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\pyPhoTimeline\pypho_timeline\core\time_synchronized_plotter_base.py:3: PythonQtWarning: Selected binding 'pyqt5' could not be found; falling back to 'pyqt6'
  from qtpy import QtCore, QtWidgets
H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\NeuroPy\neuropy\utils\mixins\time_slicing.py:405: UserWarning: registration of accessor <class 'neuropy.utils.mixins.time_slicing.TimePointEventAccessor'> under name 'time_point_event' for type <class 'pandas.core.frame.DataFrame'> is overriding a preexisting attribute with the same name.
  class TimePointEventAccessor(TimeColumnAliasesProtocol, TimeSlicableObjectProtocol, DataframeMetadataProtocol):


Using matplotlib as 2D backend.


C:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\PhoPyMNEHelper\src\phopymnehelper\helpers\indexing_helpers.py:1080: UserWarning: registration of accessor <class 'phopymnehelper.helpers.indexing_helpers.PhoDataframeAccessor'> under name 'pho' for type <class 'pandas.core.frame.DataFrame'> is overriding a preexisting attribute with the same name.
  class PhoDataframeAccessor:


2026-04-02 10:24:50 - pypho_timeline - INFO - [MainThread] - Logging configured: level=DEBUG, console=True, file=False, log_file=N/A


In [ ]:
## 

### Motion track: BAD intervals (exclude + dark overlay)

`MotionTrackDatasource` accepts optional bad segments aligned with `motion_df['t']` (Unix seconds or datetimes in the same time base as the track).

- **`bad_intervals_df`**: either columns `t_start` + `t_duration`, or MNE-style `onset` + `duration` (then pass **`bad_intervals_time_origin_unix`** = recording start as Unix seconds so `t_start = origin + onset`).
- **`exclude_bad_from_detail`** (default `True`): rows whose time falls inside a bad interval are dropped **before** downsampling.
- **`bad_overlay_alpha`** (default `0.9`): vertical black bands drawn on top of the motion lines via `LinearRegionItem`.

Helpers: `normalize_motion_bad_intervals_df`, `motion_bad_intervals_key_suffix`.

**PhoPyMNEHelper `is_moving_annots_df`** (has `onset`, `duration` in seconds relative to the recording):

```python
from pypho_timeline.rendering.datasources.specific.motion import MotionTrackDatasource, normalize_motion_bad_intervals_df

recording_start_unix = float(your_meas_date.timestamp())  # same origin as timeline motion `t`
ds = MotionTrackDatasource(
    intervals_df, motion_df,
    bad_intervals_df=is_moving_annots_df,
    bad_intervals_time_origin_unix=recording_start_unix,
    exclude_bad_from_detail=True,
    bad_overlay_alpha=0.9,
)
```

If bad intervals are already in absolute Unix time, use `bad_intervals_df=normalize_motion_bad_intervals_df(df)` and omit `bad_intervals_time_origin_unix`.

After **`set_bad_intervals(...)`** on an already displayed track, refresh the cached renderer if needed: `track_renderer.detail_renderer = ds.get_detail_renderer()` (and re-trigger a viewport refresh / overview update as your app does for `source_data_changed_signal`).

In [2]:
# n_most_recent_sessions_to_preprocess: int = None # None means all sessions
# n_most_recent_sessions_to_preprocess: int = 100
# n_most_recent_sessions_to_preprocess: int = 15
n_most_recent_sessions_to_preprocess: int = 5
# n_most_recent_sessions_to_preprocess: int = 3
# n_most_recent_sessions_to_preprocess = None

# enable_video_track: bool = True
enable_video_track: bool = False

# Optional: only include streams whose name matches any of these patterns (regex).
STREAM_ALLOWLIST: Optional[List[str]] = None  # e.g. [r"EEG.*", r"MOTION.*"]

# Optional: exclude streams whose name matches any of these patterns (regex).
# STREAM_BLOCKLIST: Optional[List[str]] = ['Epoc X Motion', 'Epoc X eQuality']
STREAM_BLOCKLIST: Optional[List[str]] = ['Epoc X eQuality']


# db_root_path = Path('/content/drive/MyDrive/Databases')
# db_root_path = Path(r'E:/Dropbox (Personal)/Databases') ## APOGEE
db_root_path = Path(r'E:/Dropbox (Personal)/Databases') # WIN10_VM
assert db_root_path.exists(), f"'{db_root_path.as_posix()}' does not exist!"

pho_log_to_LSL_recordings_path: Path = db_root_path.joinpath('UnparsedData/PhoLogToLabStreamingLayer_logs')
## These contain little LSL .fif files with names like: '20250808_062814_log.fif',

video_recordings_path: Path = db_root_path.joinpath('UnparsedData/LabRecorderStudies/sub-P001/Videos')

# eeg_analyzed_parent_export_path = db_root_path.joinpath('AnalysisData/MNE_preprocessed')
# pickled_data_path = db_root_path.joinpath('AnalysisData/MNE_preprocessed/PICKLED_COLLECTION')
# assert pickled_data_path.exists()
xdf_to_rerun_rrd_parent_export_path = db_root_path.joinpath('AnalysisData/to_rerun_rrd').resolve()
xdf_to_rerun_rrd_parent_export_path.mkdir(exist_ok=True)
# print(f'xdf_to_rerun_rrd_parent_export_path: "{xdf_to_rerun_rrd_parent_export_path.as_posix()}"')

# lab_recorder_output_path = Path(r"E:\Dropbox (Personal)\Databases\UnparsedData\LabRecorderStudies\sub-P001")
lab_recorder_output_path = db_root_path.joinpath('UnparsedData/LabRecorderStudies/sub-P001')
assert lab_recorder_output_path.exists()

xdf_file_cache_filename: str = f"{get_now_time_str()}_found_xdf_files.csv"
xdf_file_cache_filepath: Path = xdf_to_rerun_rrd_parent_export_path.joinpath(xdf_file_cache_filename).resolve()
print(f'exporting xdf .csv to xdf_file_cache_filepath: "{xdf_file_cache_filepath}..."')
discovery = discover_xdf_files_for_timeline(xdf_discovery_dirs=[lab_recorder_output_path, pho_log_to_LSL_recordings_path], n_most_recent=n_most_recent_sessions_to_preprocess, csv_export_path=xdf_file_cache_filepath)
final_xdf_paths: List[Path] = discovery.xdf_paths
print(f'processing len(active_EEG_recording_files): {len(final_xdf_paths)} recording files...')

## OUTPUTS: final_xdf_paths

# ==================================================================================================================================================================================================================================================================================== #
# BEGIN MAIN                                                                                                                                                                                                                                                                           #
# ==================================================================================================================================================================================================================================================================================== #

# builder: TimelineBuilder = TimelineBuilder()
# active_video_discovery_dirs: List[Path] = [video_recordings_path] if video_recordings_path.exists() and video_recordings_path.is_dir() else []
active_video_discovery_dirs = []
builder.set_refresh_config(xdf_discovery_dirs=[lab_recorder_output_path, pho_log_to_LSL_recordings_path], n_most_recent=n_most_recent_sessions_to_preprocess, stream_allowlist=STREAM_ALLOWLIST, stream_blocklist=STREAM_BLOCKLIST, video_discovery_dirs=active_video_discovery_dirs)
timeline: SimpleTimelineWidget = builder.build_from_xdf_files(xdf_file_paths=final_xdf_paths, stream_allowlist=STREAM_ALLOWLIST, stream_blocklist=STREAM_BLOCKLIST)

if timeline is None:
    print("WARN: No streams found. Check XDF paths and stream filters.")


exporting xdf .csv to xdf_file_cache_filepath: "E:\Dropbox (Personal)\Databases\AnalysisData\to_rerun_rrd\2026-04-02_10-04_found_xdf_files.csv..."
processing len(active_EEG_recording_files): 5 recording files...


Stream 4: Calculated effective sampling rate 31.3815 Hz is different from specified rate 16.0000 Hz.
Stream 3: Calculated effective sampling rate 32.0174 Hz is different from specified rate 16.0000 Hz.
c:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\pyPhoTimeline\pypho_timeline\utils\datetime_helpers.py:233: UserWarning: Discarding nonzero nanoseconds in conversion.
  return [dt.to_pydatetime() if isinstance(dt, pd.Timestamp) else dt for dt in datetimes]
c:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\pyPhoTimeline\pypho_timeline\utils\datetime_helpers.py:233: UserWarning: Discarding nonzero nanoseconds in conversion.
  return [dt.to_pydatetime() if isinstance(dt, pd.Timestamp) else dt for dt in datetimes]


Using earliest reference datetime: 2026-04-01 13:30:25+00:00 for timestamp normalization

Processing stream 'TextLogger' from 2 file(s)...
======== STREAM "TextLogger":
	created_at_dt: 2026-04-01 05:15:12 PM
	first_timestamp_dt: 2026-04-01 05:15:12 PM
	last_timestamp_dt: 2026-04-01 05:15:12 PM
WARN: (len(phopylslhelper_dict)5
	 FOUND CUSTOM TIMESTAMP SYNC KEY: "stream_start_datetime": 2026-04-01 17:14:54+00:00
	 FOUND CUSTOM TIMESTAMP SYNC KEY: "stream_start_lsl_local_offset_seconds": 1286868.7766745
	 FOUND CUSTOM TIMESTAMP SYNC KEY: "recording_start_datetime": 2026-04-01 17:14:54+00:00
	 FOUND CUSTOM TIMESTAMP SYNC KEY: "recording_start_lsl_local_offset_seconds": 1286868.7766943
======== STREAM "EventBoard":
	WARN: skipping empty stream: "EventBoard"
======== STREAM "Epoc X eQuality":
	created_at_dt: 2026-04-01 05:15:12 PM
	first_timestamp_dt: 2026-04-01 05:15:12 PM
	last_timestamp_dt: 2026-04-01 05:15:12 PM
WARN: (len(phopylslhelper_dict)3
	 FOUND CUSTOM TIMESTAMP SYNC KEY: "stream_

c:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\pyPhoTimeline\pypho_timeline\rendering\datasources\stream_to_datasources.py:421: RuntimeWarning: Could not get size for self.info
  logger.info(f'\traws_dict: {raws_dict}')
c:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\pyPhoTimeline\pypho_timeline\rendering\datasources\stream_to_datasources.py:421: RuntimeWarning: Could not get size for self.info
  logger.info(f'\traws_dict: {raws_dict}')
c:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\pyPhoTimeline\pypho_timeline\rendering\datasources\track_datasource.py:397: FutureWarning: Using .astype to convert from timezone-aware dtype to timezone-naive dtype is deprecated and will raise in a future version.  Use obj.tz_localize(None) or obj.tz_convert('UTC').tz_localize(None) instead
  intervals_df = intervals_df.astype({'t_start_dt': 'datetime64[ns]', 't_end_dt': 'datetime64[ns]'})



Processing stream 'EventBoard' from 2 file(s)...
  Skipping empty stream from LabRecorder_Apogee_2026-04-01T171512.271Z_eeg.xdf
  Skipping empty stream from LabRecorder_Apogee_2026-04-01T133033.245Z_eeg.xdf
  No valid intervals for stream 'EventBoard'

Processing stream 'Epoc X Motion' from 2 file(s)...


...

c:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\pyPhoTimeline\pypho_timeline\rendering\datasources\stream_to_datasources.py:421: RuntimeWarning: Could not get size for self.info
  logger.info(f'\traws_dict: {raws_dict}')
c:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\pyPhoTimeline\pypho_timeline\rendering\datasources\stream_to_datasources.py:421: RuntimeWarning: Could not get size for self.info
  logger.info(f'\traws_dict: {raws_dict}')
c:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\pyPhoTimeline\pypho_timeline\rendering\datasources\track_datasource.py:397: FutureWarning: Using .astype to convert from timezone-aware dtype to timezone-naive dtype is deprecated and will raise in a future version.  Use obj.tz_localize(None) or obj.tz_convert('UTC').tz_localize(None) instead
  intervals_df = intervals_df.astype({'t_start_dt': 'datetime64[ns]', 't_end_dt': 'datetime64[ns]'})
c:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\pyPhoTimeline\pypho_timeline\rendering\datasources\stream_to_datasources.py:477: RuntimeWarning

Connected "on_options_changed" signal from options panel to track_renderer.on_options_changed
Connected "on_options_accepted" signal from options panel to track_renderer.on_options_accepted
Connected "on_options_rejected" signal from options panel to track_renderer.on_options_rejected
Connected "on_options_changed" signal from options panel to track_renderer.on_options_changed
Connected "on_options_accepted" signal from options panel to track_renderer.on_options_accepted
Connected "on_options_rejected" signal from options panel to track_renderer.on_options_rejected
Connected "on_options_changed" signal from options panel to track_renderer.on_options_changed
Connected "on_options_accepted" signal from options panel to track_renderer.on_options_accepted
Connected "on_options_rejected" signal from options panel to track_renderer.on_options_rejected
Connected "on_options_changed" signal from options panel to track_renderer.on_options_changed
Connected "on_options_accepted" signal from opti

result_df
result_df
t_duration_sec: 2358.4707939624786
curr_df_points_per_sec: 32.01778041657672
t_duration_sec: 4957.47532081604
curr_df_points_per_sec: 29.43131141517874


c:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\pyPhoTimeline\pypho_timeline\rendering\detail_renderers\log_text_plot_renderer.py:110: FutureWarning: The parsing of 'now' in pd.to_datetime without `utc=True` is deprecated. In a future version, this will match Timestamp('now') and Timestamp.now()
  for idx, row in df_sorted.iterrows():
c:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\pyPhoTimeline\pypho_timeline\rendering\detail_renderers\log_text_plot_renderer.py:110: FutureWarning: Inferring datetime64[ns] from data containing strings is deprecated and will be removed in a future version. To retain the old behavior explicitly pass Series(data, dtype=datetime64[ns])
  for idx, row in df_sorted.iterrows():


curr_downsampled_points_per_sec: 9.999699831082667
result_df


In [ ]:
timeline.go_to_latest_window() # timeline.go_to_latest_window()

In [ ]:
## this seems wrong `timeline.total_data_end_time`
(timeline.total_data_start_time, timeline.total_data_end_time) ## (datetime.datetime(2026, 4, 16, 7, 14, 0, 23324, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 4, 16, 16, 5, 55, 326405, tzinfo=datetime.timezone.utc))
(timeline.active_window_start_time, timeline.active_window_end_time) # (datetime.datetime(2026, 4, 16, 7, 14, 0, 23324, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 4, 16, 22, 57, 5, 323324, tzinfo=datetime.timezone.utc))
(timeline._last_applied_plot_window_x0, timeline._last_applied_plot_window_x1) # (1776323640.023324, 1776380225.323324)

In [ ]:
timeline.spikes_window.active_time_window

In [ ]:
timeline.spikes_window.update_window_start_end(timeline.active_window_start_time, timeline.active_window_end_time)


In [ ]:
timeline.simulate_window_scroll( 


# Compute Spectograms

In [ ]:
spec_names = [n for n in timeline.track_datasources if n.startswith("EEG_Spectrogram_")]
spec_names += [n for n in timeline.track_datasources if n.startswith("EEG_Spectrogram_") and n.endswith("__compare")]
spec_names

In [ ]:
# After: timeline = builder.build_from_xdf_files(...)
assert timeline is not None and len(timeline.get_all_track_names()) > 0, "Build the timeline first."
_names = timeline.get_all_track_names()
_eeg_names = [n for n in _names if n.startswith("EEG_")]
_motion_names = [n for n in _names if n.startswith("MOTION_")]
# if not _eeg_names:
#     raise RuntimeError(f"No EEG_* track found. Names: {_names}")
# eeg_name = _eeg_names[0]
eeg_name = 'EEG_Epoc X' ## HARDCODED
print(f'eeg_name: {eeg_name}')
motion_name = _motion_names[0] if _motion_names else None
eeg_ds = timeline.track_datasources[eeg_name]
eeg_df = eeg_ds.detailed_df.sort_values("t").reset_index(drop=True).copy()
_t_ts = eeg_df["t"]
if pd.api.types.is_datetime64_any_dtype(_t_ts):
    eeg_df["t"] = pd.to_datetime(_t_ts, utc=True, errors="coerce").astype(np.int64) / 1e9
else:
    eeg_df["t"] = pd.to_numeric(_t_ts, errors="coerce")

# eeg_ds.intervals_df
assert len(eeg_ds.raw_datasets) > 0, f"eeg datasource has no raw datasets!"
eeg_raw = eeg_ds.raw_datasets[0]
eeg_raw

In [ ]:
eeg_ds

In [ ]:
eeg_raw.annotations

In [ ]:
from phopymnehelper.analysis.computations.eeg_registry import run_eeg_computations_graph, session_fingerprint_for_raw_or_path

eeg_comps_result = run_eeg_computations_graph(eeg_raw, session=session_fingerprint_for_raw_or_path(eeg_raw), goals=("spectogram",))
spec_result = eeg_comps_result["spectogram"]
spec_result


In [ ]:
from pypho_timeline.rendering.datasources.specific.eeg import EEGSpectrogramTrackDatasource
from pypho_timeline.rendering.datasources.stream_to_datasources import default_dock_named_color_scheme_key
from pypho_timeline.docking.dock_display_configs import CustomCyclicColorsDockDisplayConfig, NamedColorScheme
from pypho_timeline.core.synchronized_plot_mode import SynchronizedPlotMode
from qtpy import QtWidgets

track_name = "EEG_Spectrogram_all"  # unique
spec_ds = EEGSpectrogramTrackDatasource(
    intervals_df=eeg_ds.intervals_df.copy(),
    spectrogram_result=spec_result,
    custom_datasource_name=track_name,
    group_config=None,
    lab_obj=eeg_ds.lab_xdf_obj,
    raw_datasets=eeg_ds.raw_datasets,
    parent=eeg_ds,
)

_scheme_key = default_dock_named_color_scheme_key(track_name)
display_config = CustomCyclicColorsDockDisplayConfig(
    named_color_scheme=NamedColorScheme[_scheme_key],
    showCloseButton=True, showCollapseButton=True, showGroupButton=True, corner_radius="3px",
)
track_widget, root_g, plot_item, dock = timeline.add_new_embedded_pyqtgraph_render_plot_widget(
    name=track_name,
    dockSize=(500, 80),
    dockAddLocationOpts=["bottom"],
    display_config=display_config,
    sync_mode=SynchronizedPlotMode.TO_GLOBAL_DATA,
)
# same x-axis setup as TimelineBuilder._add_tracks_to_timeline …
plot_item.setYRange(0, 1, padding=0)
plot_item.setLabel("left", track_name)
plot_item.hideAxis("left")
timeline.add_track(spec_ds, name=track_name, plot_item=plot_item)
track_widget.optionsPanel = track_widget.getOptionsPanel()
if hasattr(dock, "updateWidgetsHaveOptionsPanel"):
    dock.updateWidgetsHaveOptionsPanel()
getattr(timeline, "_rebuild_timeline_overview_strip", lambda: None)()

In [ ]:
spec_ds.raw_datasets

In [ ]:
def recompute(datasource: EEGSpectrogramTrackDatasource, **kwargs):
    """ a function to perform recomputation of the datasource properties at runtime 
    """
    from phopymnehelper.analysis.computations.eeg_registry import run_eeg_computations_graph, session_fingerprint_for_raw_or_path

    if len((datasource.raw_datasets or [])) < 1:
        if datasource.parent() is not None:
            if getattr(datasource.parent(), 'raw_datasets', None) is not None:
                datasource.raw_datasets = datasource.parent().raw_datasets
    eeg_raw = datasource.raw_datasets[0]

    eeg_comps_result = run_eeg_computations_graph(eeg_raw, session=session_fingerprint_for_raw_or_path(eeg_raw), goals=("spectogram",))
    spec_result = eeg_comps_result["spectogram"]
    spec_result





In [ ]:
# # modern_found_EEG_recording_files = HistoricalData.get_recording_files(recordings_dir=lab_recorder_output_path, recordings_extensions=['.xdf'])
# modern_found_EEG_recording_files = HistoricalData.get_recording_files(recordings_dir=[lab_recorder_output_path, pho_log_to_LSL_recordings_path], recordings_extensions=['.xdf']) ## both sources
# # modern_found_EEG_recording_files

# most_recent_modern_found_EEG_recording_files: List[Path] = modern_found_EEG_recording_files[:n_most_recent_sessions_to_preprocess]
# # most_recent_modern_found_EEG_recording_files


# # active_EEG_recording_files = modern_found_EEG_recording_files
# active_EEG_recording_files = most_recent_modern_found_EEG_recording_files

# print(f'processing len(active_EEG_recording_files): {len(active_EEG_recording_files)} recording files...')
# most_recent_modern_found_EEG_recording_file_df: pd.DataFrame = HistoricalData.build_file_comparison_df(recording_files=active_EEG_recording_files) ## 8m for 65 files
# # most_recent_modern_found_EEG_recording_file_df: pd.DataFrame = HistoricalData.build_file_comparison_df(recording_files=most_recent_modern_found_EEG_recording_files) ## 5m for 35 files !! SLOWER: 9.2min for 35 files
# most_recent_modern_found_EEG_recording_file_df


# # modern_found_EEG_recording_file_df: pd.DataFrame = HistoricalData.build_file_comparison_df(recording_files=modern_found_EEG_recording_files)
# # modern_found_EEG_recording_file_df

# ## OUTPUTS: modern_found_EEG_recording_file_df, modern_found_EEG_recording_files

# ## INPUTS: most_recent_modern_found_EEG_recording_file_df

# xdf_file_cache_filename: str = f"{get_now_time_str()}_found_xdf_files.csv"
# xdf_file_cache_filepath: Path = xdf_to_rerun_rrd_parent_export_path.joinpath(xdf_file_cache_filename).resolve()

# print(f'exporting xdf .csv to xdf_file_cache_filepath: "{xdf_file_cache_filepath}..."')
# most_recent_modern_found_EEG_recording_file_df.to_csv(xdf_file_cache_filepath)


# most_recent_modern_found_EEG_recording_file_df['src_file'].to_list()
# final_xdf_paths: List[Path] = [Path(v) for v in most_recent_modern_found_EEG_recording_file_df['src_file'].to_list()]
# final_xdf_paths

## OUTPUTS: final_xdf_paths



# active_xdf_path: List[Path] = [
#     # Path(r"E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-05T022422.773Z_eeg.xdf").resolve(),
#     # Path(r"E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-12-24T213732.785Z_eeg.xdf").resolve(),
#     # Path(r"E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-05T022422.773Z_eeg.xdf").resolve(),
#     # Path(r"E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-09T233324.449Z_eeg.xdf").resolve(), ## COGNITION WAS ABYSMAL, most sleep debt in a single week, only 3-day-binge.

#     # Path("E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-21T222456.756Z_eeg.xdf"),
#     # Path("E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-21T222210.952Z_eeg.xdf"),
#     # Path("E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-21T221511.412Z_eeg.xdf"),
#     # Path("E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-22T194023.711Z_eeg.xdf"),
#     # Path("E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-22T134325.495Z_eeg.xdf"),
#     # Path("E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_rMBPPinkDot_2026-01-22T022657.502Z.xdf"),
#     # Path("E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-22T023809.607Z_eeg.xdf"),

#     # # 2026-03-01 _________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________ #
#     # Path('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-03-01T021033.491Z_eeg.xdf'),
#     # Path('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260301_020924_log.xdf'),
#     # Path('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260301_015939_log.xdf'),
#     # Path('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-02-28T152439.698Z_eeg.xdf'),
#     # Path('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260228_152426_log.xdf'),

#     # 2026-03-04 _________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________ #
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-03-04T225210.949Z_eeg.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260304_225204_log.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-03-04T192035.507Z_eeg.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260304_192023_log.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-03-04T191528.965Z_eeg.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260304_191511_log.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-03-04T162633.469Z_eeg.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260304_162623_log.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-03-03T223004.805Z_eeg.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260303_222952_log.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-03-03T191723.860Z_eeg.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260303_191710_log.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-03-03T012438.863Z_eeg.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-03-03T005759.073Z_eeg.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260303_005724_log.xdf'),
    
# ]


# active_xdf_path: List[Path] = final_xdf_paths


# ## INPUTS: active_xdf_path
# timeline: SimpleTimelineWidget = builder.build_from_xdf_files(xdf_file_paths=active_xdf_path,
#     # stream_allowlist=[r"EEG.*", r"MOTION.*"],
#     #  stream_allowlist=[r'Epoc X*', r'Epoc X Motion*'], # , 'WhisperLiveLogger', 'Epoc X eQuality', 'TextLogger', 'EventBoard'
#     # stream_allowlist=[r'Epoc X', r'Epoc X Motion*'], # , 'WhisperLiveLogger', 'Epoc X eQuality', 'TextLogger', 'EventBoard'
#     # stream_blocklist=['WhisperLiveLogger', 'Epoc X eQuality', 'TextLogger', 'EventBoard']
#     stream_blocklist=STREAM_BLOCKLIST,
# )
# # assert demo_xdf_path.exists()


### ADHD / theta–delta sleep intrusion (`compute_theta_delta_sleep_intrusion_series`)

Run **after** the timeline is built (previous cell). Resolves `EEG_*` and optional `MOTION_*` tracks, builds `mne.io.RawArray` from full-rate `detailed_df`, aligns motion times to `raw.times`, and stashes `adhd_ctx` for the next cells.

In [ ]:
from datetime import datetime, timezone
import mne
import numpy as np
import pandas as pd
from phopymnehelper.analysis.computations.specific import compute_theta_delta_sleep_intrusion_series
from phopymnehelper.analysis.computations.specific.ADHD_sleep_intrusions import apply_adhd_sleep_intrusion_to_timeline


# ['LOG_TextLogger',
#  'MOTION_Epoc X Motion',
#  'EEG_Spectrogram_Epoc X_Frontal-L',
#  'EEG_Spectrogram_Epoc X_Frontal-R',
#  'EEG_Spectrogram_Epoc X_Posterior-L',
#  'EEG_Spectrogram_Epoc X_Posterior-R',
#  'EEG_Epoc X']


# After: timeline = builder.build_from_xdf_files(...)
assert timeline is not None and len(timeline.get_all_track_names()) > 0, "Build the timeline first."
_names = timeline.get_all_track_names()
_eeg_names = [n for n in _names if n.startswith("EEG_")]
_motion_names = [n for n in _names if n.startswith("MOTION_")]
# if not _eeg_names:
#     raise RuntimeError(f"No EEG_* track found. Names: {_names}")
# eeg_name = _eeg_names[0]
eeg_name = 'EEG_Epoc X' ## HARDCODED
print(f'eeg_name: {eeg_name}')
motion_name = _motion_names[0] if _motion_names else None
eeg_ds = timeline.track_datasources[eeg_name]
eeg_df = eeg_ds.detailed_df.sort_values("t").reset_index(drop=True).copy()
_t_ts = eeg_df["t"]
if pd.api.types.is_datetime64_any_dtype(_t_ts):
    eeg_df["t"] = pd.to_datetime(_t_ts, utc=True, errors="coerce").astype(np.int64) / 1e9
else:
    eeg_df["t"] = pd.to_numeric(_t_ts, errors="coerce")
t0 = float(eeg_df["t"].iloc[0])
_ch = list(eeg_ds.channel_names) if getattr(eeg_ds, "channel_names", None) else [c for c in eeg_df.columns if c != "t"]
_ch = [c for c in _ch if c in eeg_df.columns]
_dt = float(np.median(np.diff(eeg_df["t"].to_numpy(dtype=float))))
sfreq = (1.0 / _dt) if _dt > 0 else 256.0
assume_microvolts = True
_data = eeg_df[_ch].to_numpy(dtype=float).T
if assume_microvolts:
    _data = _data * 1e-6
raw_adhd = mne.io.RawArray(_data, mne.create_info(ch_names=list(_ch), sfreq=float(sfreq), ch_types="eeg"))
raw_adhd.set_meas_date(datetime.fromtimestamp(t0, tz=timezone.utc))
motion_df_adhd = None
if motion_name is not None:
    _mdf = timeline.track_datasources[motion_name].detailed_df.sort_values("t").copy()
    _mt = _mdf["t"]
    if pd.api.types.is_datetime64_any_dtype(_mt):
        _mdf["t"] = pd.to_datetime(_mt, utc=True, errors="coerce").astype(np.int64) / 1e9
    else:
        _mdf["t"] = pd.to_numeric(_mt, errors="coerce")
    _mdf["t"] = _mdf["t"].to_numpy(dtype=float) - t0
    motion_df_adhd = _mdf
adhd_ctx = dict(eeg_name=eeg_name, motion_name=motion_name, eeg_ds=eeg_ds, eeg_df=eeg_df, t0=t0, raw=raw_adhd, motion_df=motion_df_adhd, out=None)
print("eeg_name=", eeg_name, "motion_name=", motion_name, "sfreq=", sfreq, "ch=", len(_ch))

In [ ]:
# Synchronous run (blocks the kernel until done). For GUI threading see the next cell.
out_adhd = compute_theta_delta_sleep_intrusion_series(adhd_ctx["raw"], motion_df=adhd_ctx["motion_df"], window_sec=4.0, step_sec=1.0)
adhd_ctx["out"] = out_adhd
print("session_mean_theta_delta", out_adhd["session_mean_theta_delta"], "valid", out_adhd["n_valid_windows"], "/", out_adhd["n_windows"])

import matplotlib.pyplot as plt
_x = adhd_ctx["t0"] + out_adhd["times"]
plt.figure(figsize=(10, 2.2))
plt.plot(_x, out_adhd["theta_delta_ratio"], color="tab:purple", linewidth=0.8)
plt.xlabel("Time (unix s)")
plt.ylabel("theta / delta")
plt.title("ADHD sleep intrusion series (NaN = motion/QC excluded)")
plt.tight_layout()
plt.show()

In [ ]:

apply_adhd_sleep_intrusion_to_timeline(timeline, adhd_ctx)

# Optional: run compute off the GUI thread, then marshal UI updates to the Qt main thread:
# def run_adhd_in_background():
#     from phopymnehelper.analysis.computations.specific import compute_theta_delta_sleep_intrusion_series
#     fut = ThreadPoolExecutor(max_workers=1).submit(lambda: compute_theta_delta_sleep_intrusion_series(adhd_ctx["raw"], motion_df=adhd_ctx["motion_df"], window_sec=4.0, step_sec=1.0))
#     def _done(f):
#         try:
#             adhd_ctx["out"] = f.result()
#         except Exception as exc:
#             print("ADHD compute failed:", exc)
#             return
#         QtCore.QTimer.singleShot(0, lambda: apply_adhd_sleep_intrusion_to_timeline(timeline, adhd_ctx))
#     fut.add_done_callback(_done)
# run_adhd_in_background()

In [ ]:
from phopymnehelper.analysis.computations.specific.bad_epochs import compute_bad_epochs_qc, apply_bad_epochs_overlays_to_timeline

out = compute_bad_epochs_qc(adhd_ctx["raw"], use_autoreject=True, copy_raw=True)
apply_bad_epochs_overlays_to_timeline(timeline, out, time_offset=t0)

In [ ]:
timeline.hide_extra_xaxis_labels_and_axes()

In [ ]:
timeline.get_all_track_names()


## Futher Timeline Widget/UI Customization

In [3]:
from pypho_timeline.widgets.timeline_overview_strip import TimelineOverviewStrip
from pypho_timeline.EXTERNAL.pyqtgraph_extensions.graphicsObjects.CustomLinearRegionItem import CustomLinearRegionItem

strip: TimelineOverviewStrip = timeline.ui.timeline_overview_strip
region: CustomLinearRegionItem = strip._viewport_region
# strip._intervals_item.setEnabled(True)
# strip._intervals_item.movable = True
# region.setMovable(True) #.movable

curr_downsampled_points_per_sec: 9.999848066180533
result_df


t_duration_sec: 2358.5860657691956
curr_df_points_per_sec: 128.37988165644944
t_duration_sec: 4934.445882797241
curr_df_points_per_sec: 118.59629508557049
curr_downsampled_points_per_sec: 9.999635095914266
result_df
curr_downsampled_points_per_sec: 9.999907015299527
result_df
mask_col 'is_valid' not found; available columns: ['AccX', 'AccX_diff', 'AccX_smooth', 'AccY', 'AccY_diff', 'AccY_smooth', 'AccZ', 'AccZ_diff', 'AccZ_smooth', 'GyroX', 'GyroY', 'GyroZ', 'is_moving', 't', 'total']
mask_col 'is_valid' not found; available columns: ['AccX', 'AccX_diff', 'AccX_smooth', 'AccY', 'AccY_diff', 'AccY_smooth', 'AccZ', 'AccZ_diff', 'AccZ_smooth', 'GyroX', 'GyroY', 'GyroZ', 'is_moving', 't', 'total']
mask_col 'is_valid' not found; available columns: ['AF3', 'AF4', 'F3', 'F4', 'F7', 'F8', 'FC5', 'FC6', 'O1', 'O2', 'P7', 'P8', 'T7', 'T8', 't']


In [ ]:
# strip

strip_active_window_float_ts = region.getRegion() # (1776323640.023324, 1776370487.5812337)
strip_active_window_float_ts # (1775008763.0, 1775065348.2999997)


In [ ]:
new_active_window_float_ts = (1776323640.023324, 1776370487.5812337)
new_active_window_float_ts


In [ ]:
region.setRegion(new_active_window_float_ts)


In [ ]:
# timeline.apply_active_window_from_plot_x(*strip_active_window_float_ts, block_signals=False)
# timeline.apply_active_window_from_plot_x(*new_active_window_float_ts, block_signals=True)
timeline.apply_active_window_from_plot_x(*(1775008763.0, 1775065348.2999997), block_signals=True)


In [ ]:
timeline.get_plot_view_range()

In [ ]:
# strip._intervals_item.data
latest_item_ts_float: float = strip._intervals_item.boundingRect().right() ## get right edge timestamp - 1775751404.0
curr_region_width_float: float = np.diff(region.getRegion())[0] # 43214.56095266342
curr_region_width_float

new_window_from_end = [(latest_item_ts_float-curr_region_width_float), latest_item_ts_float]
new_window_from_end

region.setRegion(new_window_from_end)
# strip.set_viewport(new_window_from_end)

In [ ]:
timeline.get_track_names_for_window_sync_group('primary')[0]

a_widget, _, _ = timeline.get_track_tuple(timeline.get_track_names_for_window_sync_group('primary')[0])
a_widget.on_window_changed(start_t=1776323640.023324, end_t=1776370487.5812337)

In [ ]:
motion_widget, motion_track, motion_ds = timeline.get_track_tuple('MOTION_Epoc X Motion')
motion_widget.on_window_changed(start_t=1776323640.023324, end_t=1776370487.5812337)

In [ ]:
region.setAcceptHoverEvents(True)
region.setAcceptTouchEvents(True)

In [ ]:
# region.setRegion()
region.getRegion() # (1776323640.023324, 1776370487.5812337)



In [ ]:
timeline.total_data_end_time

In [ ]:
strip.setAcceptDrops(True)
strip.setInteractive(True)
strip.setMouseTracking(True)
strip.setMouseEnabled(True)

In [ ]:
vb = strip.getViewBox()
vb.setMouseEnabled(False, False)

In [ ]:
a_widget, a_renderer, a_ds = timeline.get_track_tuple('LOG_TextLogger')

In [ ]:
rendered_items = list(a_renderer.detail_graphics.values())[0]

rendered_text_items: list[pg.TextItem] = [v for v in rendered_items if isinstance(v, pg.TextItem)]
rendered_text_items


In [ ]:
for a_text_item in rendered_text_items:
    a_text_item.setAnchor([0.5, 0.5])
    a_text_item.set

In [ ]:
from pypho_timeline.rendering.datasources.specific import EEGTrackDatasource


a_ds: EEGTrackDatasource = timeline.track_datasources['EEG_Epoc X']
a_ds

In [ ]:
a_ds.intervals_df

In [ ]:

timeline = builder.build_from_xdf_file(demo_xdf_path)


In [ ]:
# timeline = main_all_modalities_from_xdf_file_example(demo_xdf_path)

# Motion Bad Period Detection

In [ ]:
timeline.get_all_track_names()

In [ ]:
eeg_spectogram_track_names = ['EEG_Spectrogram_Epoc X_Frontal-L',
 'EEG_Spectrogram_Epoc X_Frontal-R',
 'EEG_Spectrogram_Epoc X_Posterior-L',
 'EEG_Spectrogram_Epoc X_Posterior-R',]

for a_spectogram_track_name in eeg_spectogram_track_names:
    a_spectogram_widget, a_spectogram_track, a_spectogram_ds = timeline.get_track_tuple(a_spectogram_track_name)
    # a_spectogram_track
    a_root_graphics_layout_widget = a_spectogram_widget.getRootGraphicsLayoutWidget()
    a_plot_item = a_spectogram_widget.getRootPlotItem()
    a_plot_item.hideAxis('bottom')
    # a_spectogram_ds.detailed_df

In [ ]:
motion_track = timeline.get_track('MOTION_Epoc X Motion')
motion_track

In [ ]:
from phopymnehelper.motion_data import MotionData, MotionDataFrame, BadMotionDataFrame
from phopymnehelper.MNE_helpers import MNEHelpers

motion_widget, motion_track, motion_ds = timeline.get_track_tuple('MOTION_Epoc X Motion')


In [ ]:
total_accel_threshold: float = 0.5

# a_motion_df: pd.DataFrame = motion_ds.detailed_df.copy() # a_ds.to_data_frame()       
# a_motion_df: pd.DataFrame = MotionData.compute_rolling_motion_change_detection(a_df=a_motion_df, total_change_threshold=total_accel_threshold, enable_global_normalization=True)
is_moving_annots, is_moving_annots_df = MotionData.find_high_accel_periods(a_ds=motion_ds.detailed_df, total_change_threshold=total_accel_threshold, 
    should_set_bad_period_annotations=False, minimum_bad_duration=0.050) # at least 50ms in duration to prevent tons of tiny intervals

motion_ds.detailed_df


In [ ]:
## How can I mask an existing track (such as the Motion Track) using a set of BAD_INTERVALS like is_moving_annots_df: pd.DataFrame? I'd like the data object to be set to indicate that these periods should be excluded, and the BAD periods to be rendered darkened by overlaying a BLACK fill with 90% opacity for the regions
# motion_track.set_bad_intervals(bad_intervals_df=is_moving_annots_df)
motion_ds.set_bad_intervals(bad_intervals_df=is_moving_annots_df)

In [ ]:
from pypho_timeline._embed.interval_datasource import IntervalsDatasource
from pypho_timeline.rendering.graphics.interval_rects_item import IntervalRectsItem, IntervalRectsItemData
from pypho_timeline.rendering.helpers.render_rectangles_helper import Render2DEventRectanglesHelper

# motion_ds._bad_intervals_unix_df
# _detail_t_column_to_unix_numpy
# motion_ds._bad_intervals_unix_df['t_start'] = motion_ds._bad_intervals_unix_df['onset']

motion_ds._bad_intervals_unix_df = motion_ds._bad_intervals_unix_df.bad_motion_epochs_df.adding_unix_float_columns(inplace=False)
# motion_ds.bad_intervals_unix_df
print(motion_ds._bad_intervals_unix_df.dtypes)
# onset           datetime64[ns]
# duration       timedelta64[ns]
# description             object
# t_start                float64
# t_duration             float64


In [ ]:
motion_ds._bad_intervals_unix_df

In [ ]:
bad_motion_intervals_ds: IntervalsDatasource = IntervalsDatasource(df=motion_ds._bad_intervals_unix_df, datasource_name='bad_motion_epochs')
bad_motion_intervals_ds


In [ ]:
motion_widget._update_plots()

In [ ]:
# motion_widget
_out_bad_motion_rendered_intervals = timeline.add_rendered_intervals(interval_datasource=motion_ds._bad_intervals_unix_df, name='bad_motion_epochs',
                                                child_plots=[a_plot_item],
                                                debug_print=True)

# motion_track.add_render

In [ ]:
plot_item = _out_bad_motion_rendered_intervals['RootPlot']['plot']
rect_item: IntervalRectsItem = _out_bad_motion_rendered_intervals['RootPlot']['rect_item']
rect_item

In [ ]:
plot_item.zValue()
rect_item.zValue()
# plot_item.children()

rect_item.setZValue(50)


In [ ]:
an_IntervalRectsItem: IntervalRectsItem = Render2DEventRectanglesHelper.build_IntervalRectsItem_from_interval_datasource(interval_datasource=bad_motion_intervals_ds)

motion_

In [ ]:
motion_track.

In [ ]:
timeline.hide_extra_xaxis_labels_and_axes()

In [ ]:
motion_track.visible_intervals

## End motion annotations

In [ ]:
hasattr(motion_track, "channels_enabled")
motion_track.update_channel_visibility(channel_name='GyroX', is_visible=False)
motion_track.update_channel_visibility(channel_name='GyroY', is_visible=False)
motion_track.update_channel_visibility(channel_name='GyroZ', is_visible=False)

In [ ]:
motion_detail_renderer = motion_track.detail_renderer
motion_detail_renderer.normalize = False

motion_track.datasource.detailed_df
# motion_detail_renderer
# motion_detail_renderer.motion_df


In [ ]:
## INPUTS: timeline
# Get the "EEG Epoc X" track from the timeline
# Fallback to searching tracks by attribute, as there is no `.get_track_by_name` method
# eeg_track = timeline.get_track('EEG_Epoc X')
eeg_widget, eeg_track, eeg_ds = timeline.get_track_tuple('EEG_Epoc X')
detailed_eeg_df: pd.DataFrame = eeg_ds.detailed_df
# print(detailed_eeg_df.columns) # ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4', 't']
detailed_eeg_df




In [ ]:
motion_ds._bad_intervals_unix_df = motion_ds._bad_intervals_unix_df.bad_motion_epochs_df.adding_unix_float_columns(inplace=False)
mask_bad_intervals_df: pd.DataFrame = motion_ds._bad_intervals_unix_df.copy()
mask_bad_intervals_df

In [ ]:
from phopymnehelper.helpers.dataframe_accessor_helpers import MaskedValidDataFrameAccessor

# mask_by_intervals

## find if each detailed_eeg_df['t'] is within the bad interval
# detailed_eeg_df['t']
# Efficiently see if each `detailed_eeg_df['t']` value falls within `mask_bad_intervals_df[['onset', 'onset_end']]` where all columns are datetime64[ns] 

# mask_bad_intervals_df[['onset', 'onset_end']]
# detailed_eeg_df['t'].dtypes # dtype('<M8[ns]')

detailed_eeg_df = detailed_eeg_df.masked_df.masking_by_intervals(mask_bad_intervals_df=mask_bad_intervals_df, time_col_name='t', bool_mask_column_name='is_bad_motion',
                                                            intervals_start_col_name='onset', intervals_end_col_name='onset_end')
detailed_eeg_df.masked_df.add_masking_column(mask_col='is_valid', value_cols=['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4'])
detailed_eeg_df


In [ ]:
detailed_eeg_df

In [ ]:
eeg_track.

In [ ]:


masked_detailed_eeg_df: pd.DataFrame = detailed_eeg_df.masked_df.get_masked(copy=True)

# masked_detailed_eeg_df: pd.DataFrame = detailed_eeg_df.masked_df.add_masking_column(mask_col='is_valid', value_cols=['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4'],
#                                                      copy=True)
masked_detailed_eeg_df

In [ ]:
masked_detailed_eeg_df.masked_df.get_masked()

In [ ]:
detailed_eeg_df = detailed_eeg_df.drop(columns=['is_bad_motion_right', 'is_bad_motion'], inplace=False)

In [ ]:

# ii = pd.IntervalIndex.from_arrays(
#     mask_bad_intervals_df["onset"],
#     mask_bad_intervals_df["onset_end"],
#     closed="both",  # or "left" / "right" / "neither"
# )
# in_bad = ii.contains(detailed_eeg_df["t"].values)


# import polars as pl

# points = pl.from_pandas(detailed_eeg_df).lazy()  # or scan_csv / DataFrame
# intervals = pl.from_pandas(mask_bad_intervals_df).select("onset", "onset_end").lazy()

# hits = points.join_where(
#     intervals,
#     pl.col("t").is_between(pl.col("onset"), pl.col("onset_end")),
# )

# # timestamps that fall in at least one bad interval
# in_bad = (
#     hits.select("t")
#     .unique()
#     .with_columns(in_bad=pl.lit(True))
# )

# out = (
#     points.join(in_bad, on="t", how="left")
#     .with_columns(pl.col("in_bad").fill_null(False))
# )

# (pl.col("t") >= pl.col("onset")) & (pl.col("t") < pl.col("onset_end"))

import polars as pl

lf = pl.from_pandas(detailed_eeg_df).lazy()
iv = pl.from_pandas(mask_bad_intervals_df).select("onset", "onset_end").lazy()

bad_t = (
    lf.join(iv, how="cross")
    .filter((pl.col("t") >= pl.col("onset")) & (pl.col("t") <= pl.col("onset_end")))
    .select("t")
    .unique()
    .with_columns(pl.lit(True).alias("is_bad_motion"))
)

df = lf.join(bad_t, on="t", how="left").with_columns(
    pl.col("is_bad_motion").fill_null(False)
).collect()

detailed_eeg_df = df.to_pandas()  # if you need pandas again
detailed_eeg_df


In [ ]:
# eeg_track = timeline.get_track('MOTION_Epoc X Motion')

# if hasattr(timeline, "tracks"):
#     for tr in timeline.tracks:
#         # Try 'name' or 'track_name' attribute, fallback to 'datasource.name'
#         track_name = getattr(tr, "name", None) or getattr(tr, "track_name", None)
#         if track_name == "EEG Epoc X":
#             eeg_track = tr
#             break
#         elif hasattr(tr, "datasource") and hasattr(tr.datasource, "name"):
#             if tr.datasource.name == "EEG Epoc X":
#                 eeg_track = tr
#                 break
# eeg_track
eeg_track.channel_visibility

In [ ]:
hasattr(eeg_track, "channels_enabled")
eeg_track.update_channel_visibility(channel_name='GyroX', is_visible=False)


In [ ]:
channels_to_hide = [] # ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']
channels_to_hide = ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'FC6', 'F4', 'F8', 'AF4'] # , 'O1', 'O2', 'P8', 'T8'
for a_ch in channels_to_hide:
    eeg_track.update_channel_visibility(channel_name=a_ch, is_visible=False)
# eeg_track.update_channel_visibility(channel_name='P8', is_visible=False)


In [ ]:
print(list(eeg_track.channel_visibility.keys()))

In [ ]:
eeg_track.visible_intervals ## this should not be empty


In [ ]:


eeg_track._trigger_visibility_render()

In [ ]:

if eeg_track is not None and hasattr(eeg_track, "channel_visibility"):
    # If there's an attribute listing enabled channels, we'll disable half
    enabled_channels = eeg_track.channel_visibility
    # Otherwise fall back to all channel names if present
    if not enabled_channels and hasattr(eeg_track, "all_channel_names"):
        enabled_channels = eeg_track.all_channel_names
    if enabled_channels:
        # Compute half the channels (disable the second half)
        half = len(enabled_channels) // 2
        # Set only the first half as enabled
        eeg_track.channel_visibility = enabled_channels[:half]
        # If the track supports setting enabled/disabled channels via a method, use it
        if hasattr(eeg_track, 'set_enabled_channels'):
            eeg_track.set_enabled_channels(enabled_channels[:half])
        # Potentially trigger an update/redraw
        if hasattr(eeg_track, 'update_channel_visibility'):
            eeg_track.update_channel_visibility()
else:
    print("Could not find 'EEG Epoc X' track or it doesn't support channel control.")


# Timeline Widget from just Video Track

In [ ]:
enable_video_track: bool = True ## always true for run-video-only mode

In [4]:
from pathlib import Path
from phopylslhelper.file_metadata_caching.manager import BaseFileMetadataManager
from phopylslhelper.file_metadata_caching.video_metadata import VideoMetadataParser
from phopylslhelper.file_metadata_caching.file_metadata import BaseFileMetadataParser

if enable_video_track:
    video_manager: BaseFileMetadataManager = BaseFileMetadataManager(parse_folders=[Path("M:/ScreenRecordings/EyeTrackerVR_Recordings"), Path("M:/ScreenRecordings/REC_continuous_video_recorder")],
                                                                    parsers={'video': VideoMetadataParser},)
    video_manager.metadata_df
    recent_videos = video_manager.get_most_recent_video_paths(max_num_videos=25)

In [5]:
if enable_video_track:
    # Create the VideoTrackDatasource
    video_ds: VideoTrackDatasource = VideoTrackDatasource(video_paths=recent_videos)

    # Choose a name for the new video track
    video_track_name: str = "RecentVideosTrack"

    # Add the new video track to the existing timeline
    # timeline.add_video_track(video_track_name, video_ds)
    video_widget, root_graphics, plot_item, dock = timeline.add_video_track(track_name=video_track_name, video_datasource=video_ds,
                                                                                                                    enable_time_crosshair=True,
                                                                                                                )

# timeline.add_track(video_ds, name=video_track_name)


In [ ]:
from pypho_timeline.timeline_builder import TimelineBuilder
from pypho_timeline.rendering.graphics.track_renderer import TrackRenderer

builder = TimelineBuilder()

# Build a new timeline from just video track (no XDF file needed)
# Option 1: From video folder
# video_timeline = builder.build_from_video(video_folder_path=video_folder)

# Option 2: From list of video paths
video_timeline = builder.build_from_video(video_paths=recent_videos)

# Option 3: From existing VideoTrackDatasource
# video_timeline = builder.build_from_video(video_datasource=video_ds)
video_timeline.show()

In [ ]:
video_timeline.get_all_track_names()

In [ ]:
# video_widget
video_track_name = "VideoTrack"
video_widget, video_track_renderer, video_track_datasource = video_timeline.get_track_tuple(name=video_track_name)
video_widget

In [ ]:
## Add a current datetime red line representing the current datetime (datetime.now())

video_widget.


In [ ]:
video_timeline.show()

In [ ]:
# Create a plot widget for this track
from pypho_timeline.core.synchronized_plot_mode import SynchronizedPlotMode


a_detail_renderer = video_ds.get_detail_renderer()

# The getOptionsPanel() method will be called by the dock when needed
# No need to set optionsPanel attribute if getOptionsPanel() is implemented

## if we provide a valid `optionsPanel: Optional[QWidget]` here on the widget the button will automatically show up on the dock
track_widget, a_root_graphics, a_plot_item, a_dock = timeline.add_new_embedded_pyqtgraph_render_plot_widget(
    name=video_ds.custom_datasource_name,
    dockSize=(500, 80),
    dockAddLocationOpts=['bottom'],
    sync_mode=SynchronizedPlotMode.TO_GLOBAL_DATA
)

assert a_detail_renderer is not None
track_widget.set_track_renderer(a_detail_renderer)
# Explicitly set the attribute (not just rely on getOptionsPanel())
track_widget.optionsPanel = track_widget.getOptionsPanel()

# Try to force dock to update button visibility
a_dock.updateWidgetsHaveOptionsPanel()

a_dock.update()  # May refresh the title bar
# Or if available:
if hasattr(a_dock, 'updateTitleBar') or hasattr(a_dock, 'refresh'):
    a_dock.updateTitleBar()  # or refresh()



# Set the plot to show the full time range
a_plot_item.setXRange(
    timeline.total_data_start_time, 
    timeline.total_data_end_time, 
    padding=0
)
a_plot_item.setYRange(0, 1, padding=0)
a_plot_item.setLabel('bottom', 'Time', units='s')
a_plot_item.setLabel('left', video_ds.custom_datasource_name)
a_plot_item.hideAxis('left')  # Hide Y-axis for cleaner look

# Add the track to the plot
a_track_name: str = video_ds.custom_datasource_name
timeline.add_track(
    video_ds,
    name=a_track_name,
    plot_item=a_plot_item
)

# Messing with stream:

In [ ]:
a_track_name: str = 'MOTION_Epoc X Motion'
# a_track_name: str = 'EEG_Epoc X'
a_renderer = timeline.track_renderers[a_track_name]
a_detail_renderer = a_renderer.detail_renderer # MotionPlotDetailRenderer 
a_ds = timeline.track_datasources[a_track_name]
# interval = a_ds.intervals_df
interval = a_ds.get_overview_intervals()
# interval = None
type(interval)
interval

In [ ]:
a_renderer.visible_intervals
a_renderer.detail_graphics
a_renderer.overview_rects_item



In [ ]:
from pypho_timeline.core.pyqtgraph_time_synchronized_widget import PyqtgraphTimeSynchronizedWidget

dDisplayItem = timeline.ui.dynamic_docked_widget_container.find_display_dock(identifier=a_track_name) # Dock
a_widget: PyqtgraphTimeSynchronizedWidget = timeline.ui.matplotlib_view_widgets[a_track_name] # PyqtgraphTimeSynchronizedWidget 
a_root_graphics_layout_widget = a_widget.getRootGraphicsLayoutWidget()
a_plot_item = a_widget.getRootPlotItem()
a_plot_item


In [ ]:
# Option 2: directly on the PlotItem (delegates to its ViewBox)
(x_min, x_max), (y_min, y_max) = a_plot_item.viewRange()

# The currently visible time window on the track:
t_start, t_end = x_min, x_max

active_viewport_duration_t: float = t_end - t_start
active_viewport_duration_t
(t_start, t_end) # (59086.148423519495, 59102.162193802214)
interval_dict_rep = {'t_start': t_start, 't_duration': active_viewport_duration_t, 't_end': t_end}
interval_dict_rep

In [ ]:
def getOptionsPanel(self):
    """Return the options panel widget for this dock item."""
    from PyQt5.QtWidgets import QWidget, QVBoxLayout, QLabel, QCheckBox, QSpinBox, QGroupBox
    
    options_widget = QWidget()
    main_layout = QVBoxLayout()
    
    # Create a group box for organization
    settings_group = QGroupBox("Settings")
    settings_layout = QVBoxLayout()
    
    # Add your actual settings controls
    self.option_checkbox = QCheckBox("Enable feature")
    settings_layout.addWidget(self.option_checkbox)
    
    self.option_spinbox = QSpinBox()
    self.option_spinbox.setRange(1, 100)
    settings_layout.addWidget(QLabel("Value:"))
    settings_layout.addWidget(self.option_spinbox)
    
    settings_group.setLayout(settings_layout)
    main_layout.addWidget(settings_group)
    main_layout.addStretch()
    
    options_widget.setLayout(main_layout)
    return options_widget

In [ ]:
a_widget._update_plots()

In [ ]:
# a_widget.options_panel
an_options_panel = a_widget.getOptionsPanel()
a_widget.optionsPanel = an_options_panel
a_widget.options_panel.show()




In [ ]:
# def _checkWidgetForOptions(self, widget):
#     """Check if a widget provides an options panel.
    
#     Returns:
#         Optional[QtWidgets.QWidget]: The options panel widget if found, None otherwise.
        
#     A widget can provide options in two ways:
#     1. Implement a method: getOptionsPanel() -> Optional[QWidget]
#     2. Provide an attribute: optionsPanel: Optional[QWidget]
#     """
#     # Check for method
#     if hasattr(widget, 'getOptionsPanel') and callable(getattr(widget, 'getOptionsPanel')):
#         try:
#             options_panel = widget.getOptionsPanel()
#             if options_panel is not None and isinstance(options_panel, QtWidgets.QWidget):
#                 return options_panel
#         except Exception as e:
#             # Silently ignore errors in getOptionsPanel
#             pass
#     # Check for attribute
#     if hasattr(widget, 'optionsPanel'):
#         options_panel = widget.optionsPanel
#         if options_panel is not None and isinstance(options_panel, QtWidgets.QWidget):
#             return options_panel
#     return None



In [ ]:
# Create the options panel
a_widget.optionsPanel = QWidget()
layout = QVBoxLayout()

# Add your options controls here
label = QLabel("Options for this widget")
layout.addWidget(label)

a_widget.optionsPanel.setLayout(layout)


In [ ]:
a_ds.detailed_df
a_ds.enable_downsampling
a_ds.max_points_per_second

a_ds.max_points_per_second


In [ ]:
a_ds.fetch_detailed_data(interval=interval_dict_rep)

In [ ]:
a_widget

In [ ]:
a_widget.active_plot_target


In [ ]:
a_ds.detailed_df

In [ ]:
graphics_objects = a_detail_renderer.render_detail(plot_item=a_plot_item, interval=interval, detail_data=a_ds.detailed_df) # List[PlotDataItem]
graphics_objects

In [ ]:
# clear_detail
a_detail_renderer.clear_detail(plot_item=a_plot_item, graphics_objects=graphics_objects)
a_detail_renderer

In [ ]:
# a_bounds_tuple = a_detail_renderer.get_detail_bounds(interval=a_ds.get_overview_intervals(), detail_data=a_ds.detailed_df) # List[PlotDataItem]
a_bounds_tuple = a_detail_renderer.get_detail_bounds(interval=interval, detail_data=a_ds.detailed_df) # List[PlotDataItem]
print(f'a_bounds_tuple: {a_bounds_tuple}')
x_min, x_max, y_min, y_max = a_bounds_tuple

In [ ]:
timeline.spikes_window.active_time_window
timeline.spikes_window.total_df_start_end_times


In [ ]:
# In your notebook, check:
track_name = 'MOTION_Epoc X Motion'
proxy_key = f'track_viewport_{track_name}'
print(f"Connection exists: {proxy_key in timeline.ui.connections}")

In [ ]:
# Check if sigRangeChanged has connections
viewbox = a_plot_item.getViewBox()
if viewbox is not None:
    print(f"sigRangeChanged connections: {viewbox.sigRangeChanged.receivers}")

In [ ]:
# Get the current viewport and trigger update
viewbox = a_plot_item.getViewBox()
if viewbox is not None:
    x_range, y_range = viewbox.viewRange()
    a_renderer.update_viewport(x_range[0], x_range[1])

## Individual parts of load from xdf file

In [ ]:
xdf_file_path = demo_xdf_path
print("=" * 60)
print(f"pyPhoTimeline - Load all EEG (or LSL) modalities from XDF: {xdf_file_path}")
print("=" * 60)

# Create Qt application
app = pg.mkQApp("pyPhoTimelineXDFExample")

# --- 1. Load the XDF file (using pyxdf) ---
import pyxdf

print(f"Loading XDF file: {xdf_file_path} ...")
streams, file_header = pyxdf.load_xdf(str(xdf_file_path))
print(f"Streams loaded: {[s['info']['name'][0] for s in streams]}")
print(f"File Header: {file_header}")

In [ ]:
all_streams, all_streams_datasources = perform_process_all_streams(streams=streams)

In [ ]:
all_streams_datasources['Epoc X Motion']


In [ ]:
# active_datasources = eeg_datasources
from pypho_timeline import SynchronizedPlotMode


active_datasources_dict = {k:v for k, v in all_streams_datasources.items() if v is not None}
active_datasources = list(active_datasources_dict.values())


# --- 4. Create the timeline widget and add EEG tracks ---
timeline = SimpleTimelineWidget(
    total_start_time=min([ds.total_df_start_end_times[0] for ds in active_datasources]),
    total_end_time=max([ds.total_df_start_end_times[1] for ds in active_datasources]),
    window_duration=10.0,
    window_start_time=min([ds.total_df_start_end_times[0] for ds in active_datasources]),
    add_example_tracks=False  # Don't add example tracks for XDF data
)

# Create plot widgets for each EEG stream and add tracks
for datasource in active_datasources:
    # Create a plot widget for this track
    track_widget, root_graphics, plot_item, dock = timeline.add_new_embedded_pyqtgraph_render_plot_widget(
        name=datasource.custom_datasource_name,
        dockSize=(500, 80),
        dockAddLocationOpts=['bottom'],
        sync_mode=SynchronizedPlotMode.TO_GLOBAL_DATA
    )
    
    # Set the plot to show the full time range
    plot_item.setXRange(
        timeline.total_data_start_time, 
        timeline.total_data_end_time, 
        padding=0
    )
    plot_item.setYRange(0, 1, padding=0)
    plot_item.setLabel('bottom', 'Time', units='s')
    plot_item.setLabel('left', datasource.custom_datasource_name)
    plot_item.hideAxis('left')  # Hide Y-axis for cleaner look
    
    # Add the track to the plot
    timeline.add_track(
        datasource,
        name=datasource.custom_datasource_name,
        plot_item=plot_item
    )

timeline.setWindowTitle(f"pyPhoTimeline - EEG Modalities from XDF: {xdf_file_path.name}")
timeline.resize(1000, 800)
timeline.show()

print("\nTimeline widget created with EEG tracks from XDF:")
for spec_ds in active_datasources:
    print(f"  - {spec_ds.custom_datasource_name}, time: {spec_ds.total_df_start_end_times}")

print("\nScroll on the timeline to see loaded EEG intervals for each stream.")
print("Close the window to exit.\n")


# 2026-03-18 - Smarter cross-stream post-processing

In [ ]:
## cross-stream post-processing
from pypho_timeline.rendering.datasources.specific.eeg import EEGSpectrogramTrackDatasource, EEGTrackDatasource

eeg_ds: EEGTrackDatasource = timeline.track_datasources['EEG_Epoc X']
eeg_spectogram_ds: EEGSpectrogramTrackDatasource = timeline.track_datasources['EEG_Spectrogram_Epoc X']


In [ ]:
# eeg_spectogram_ds.intervals_df
eeg_spectogram_ds.detailed_df